In [1]:
import pandas as pd
from huggingface_hub import hf_hub_download

In [2]:
train_path = hf_hub_download(repo_id = "Cleanlab/stanford-politeness", filename = "fine-tuning/train_full.csv", repo_type = "dataset")
test_path = hf_hub_download(repo_id = "Cleanlab/stanford-politeness", filename = "fine-tuning/test.csv", repo_type = "dataset")

In [3]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

In [4]:
train_df.head(10)

,text,a1,a2,a3,a4,a5,a6,a7,a8,a9,...,a214,a215,a216,a217,a218,a219,label,label_cat,label_noisy,label_cat_noisy
0,Question: how do we move the content from the ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,neutral,1,neutral
1,Why would you doubt that. And for that matter ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,impolite,0,impolite
2,Thank you for closing out this debate. Can yo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,polite,1,neutral
3,You may want to take a look at <url>. Can you...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,neutral,1,neutral
4,The consequences apparently have no sliding sc...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,impolite,0,impolite
5,"Good! By the way, what do you think of merging...",NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,neutral,2,polite
6,You didn't answer about the status of above me...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,impolite,2,polite
7,"Please take a look at Space Shuttle Discovery,...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,polite,2,polite
8,"Hi, Schutz; it's still busted. Would it be OK...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,neutral,2,polite
9,The deletion review page linked above contains...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,impolite,1,neutral


In [5]:
test_df.head()

,prompt,completion
0,Excuse me? What are you accusing me of doing?,impolite
1,I don't understand Tasc0??,impolite
2,"Well, you leaved me also curious. Why was it n...",impolite
3,"Hi, I meant to do this a long time ago, but I ...",neutral
4,DONE. Do you think everything is now sorted so...,neutral


In [6]:
train_df = train_df[["text", "label_cat", "label"]].dropna()

In [7]:
train_df.head()

,text,label_cat,label
0,Question: how do we move the content from the ...,neutral,1
1,Why would you doubt that. And for that matter ...,impolite,0
2,Thank you for closing out this debate. Can yo...,polite,2
3,You may want to take a look at <url>. Can you...,neutral,1
4,The consequences apparently have no sliding sc...,impolite,0


In [8]:
label2id = {"impolite": 0, "neutral": 1, "polite": 2}
id2label = {v: k for k, v in label2id.items()}

test_df = test_df[["prompt", "completion"]].dropna()
test_df = test_df.rename(columns={"prompt": "text", "completion": "label_cat"})
test_df["label"] = test_df["label_cat"].map(label2id)

In [9]:
test_df.head()

,text,label_cat,label
0,Excuse me? What are you accusing me of doing?,impolite,0
1,I don't understand Tasc0??,impolite,0
2,"Well, you leaved me also curious. Why was it n...",impolite,0
3,"Hi, I meant to do this a long time ago, but I ...",neutral,1
4,DONE. Do you think everything is now sorted so...,neutral,1


In [10]:
print(train_df["label_cat"].value_counts())
print(train_df["label_cat"].value_counts(normalize=True))

print(test_df["label_cat"].value_counts())
print(test_df["label_cat"].value_counts(normalize=True))

label_cat
neutral     865
impolite    689
polite      362
Name: count, dtype: int64
label_cat
neutral     0.451461
impolite    0.359603
polite      0.188935
Name: proportion, dtype: float64
label_cat
neutral     216
impolite    173
polite       91
Name: count, dtype: int64
label_cat
neutral     0.450000
impolite    0.360417
polite      0.189583
Name: proportion, dtype: float64


In [11]:
device = "cuda"

In [12]:
import torch

In [13]:
class_counts = train_df["label"].value_counts().sort_index()

In [14]:
total = class_counts.sum()

In [15]:
num_classes = len(class_counts)

In [16]:
class_weights = total / (class_counts * num_classes)
class_weights = torch.tensor(class_weights.values, dtype=torch.float)

print(class_counts)
print(class_weights)

label
0    689
1    865
2    362
Name: count, dtype: int64
tensor([0.9269, 0.7383, 1.7643])


In [17]:
from transformers import Trainer

In [18]:
import torch.nn as nn

In [19]:
class WeightedTrainer(Trainer):
  def compute_loss(self, model, inputs, return_outputs = False, **kwargs):
    labels = inputs.pop("labels")
    outputs = model(**inputs)
    logits = outputs.logits
    loss_fct = nn.CrossEntropyLoss(weight = class_weights)
    loss = loss_fct(logits.view(-1, num_classes), labels.view(-1))
    return (loss, outputs) if return_outputs else loss

In [20]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

train_ds = Dataset.from_pandas(train_df[["text", "label"]])
test_ds = Dataset.from_pandas(test_df[["text", "label"]])

lengths = train_df["text"].apply(lambda x: len(tokenizer.encode(x)))
print(lengths.describe())

count    1916.000000
mean       34.501566
std        18.308054
min         3.000000
25%        22.000000
50%        30.000000
75%        43.000000
max       179.000000
Name: text, dtype: float64


In [21]:
print((lengths > 64).mean())

0.06210855949895616


In [22]:
def tokenize_fn(example):
  tokenized = tokenizer(example["text"], truncation = True, padding = "max_length", max_length = 64)
  tokenized["labels"] = example["label"]
  return tokenized

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/1916 [00:00<?, ? examples/s]

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

In [44]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels = 3,
    id2label=id2label,
    label2id=label2id,
)

model = model.to(device)
class_weights = class_weights.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [45]:
from transformers import TrainingArguments

from google.colab import drive
drive.mount('/content/drive')

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/paraphrase-tool/checkpoints/politeness-classifier-v3",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=4,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    report_to="none",
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

In [31]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis = -1)

  p, r, f1, _ = precision_recall_fscore_support(labels, preds, average = None, labels = [1,0,2])
  acc = accuracy_score(labels, preds)
  metrics = {"accuracy" : acc}

  for i, class_name in id2label.items():
      metrics[f"f1_{class_name}"] = f1[i]
      metrics[f"precision_{class_name}"] = p[i]
      metrics[f"recall_{class_name}"] = r[i]

  return metrics


In [27]:
!pip install -U datasets

In [46]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Impolite,Precision Impolite,Recall Impolite,F1 Neutral,Precision Neutral,Recall Neutral,F1 Polite,Precision Polite,Recall Polite
1,0.801270,0.719370,0.658333,0.643564,0.691489,0.601852,0.715232,0.837209,0.624277,0.614173,0.478528,0.857143
2,0.744126,0.628466,0.675000,0.578947,0.785714,0.458333,0.788360,0.726829,0.861272,0.633333,0.510067,0.835165
3,0.551636,0.628350,0.695833,0.617564,0.795620,0.504630,0.822857,0.813559,0.832370,0.630350,0.487952,0.890110
4,0.465576,0.624199,0.725000,0.688946,0.774566,0.620370,0.818991,0.841463,0.797688,0.649573,0.531469,0.835165


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=240, training_loss=0.6728134075800578, metrics={'train_runtime': 140.7655, 'train_samples_per_second': 54.445, 'train_steps_per_second': 1.705, 'total_flos': 252062654183424.0, 'train_loss': 0.6728134075800578, 'epoch': 4.0})

In [47]:
trainer.save_model("/content/drive/MyDrive/paraphrase-tool/checkpoints/politeness-classifier-v3/final")
tokenizer.save_pretrained("/content/drive/MyDrive/paraphrase-tool/checkpoints/politeness-classifier-v3/final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/paraphrase-tool/checkpoints/politeness-classifier-v3/final/tokenizer_config.json',
 '/content/drive/MyDrive/paraphrase-tool/checkpoints/politeness-classifier-v3/final/tokenizer.json')